# Music Annotation Project 60R3
## Read-only Page258 SHAP and grouped counterfactual audit

**Runtime:** CPU is sufficient; High-RAM CPU is recommended. GPU is not required.

This notebook explains the frozen 59R1 REMOVE estimator on the already completed 60R2 Page258 output. It does not retrain, refit, recalibrate, move the 0.995 threshold, change SAFE_REMOVE, or overwrite any 60R2 or 59R1 artefact.

### Questions answered

1. Which predictors push the frozen raw REMOVE score downward on Page258 pixels that remain in `M5_REMOVE`?
2. How much of that downward pressure is associated with `p_print` versus boundary, protected-proximity, topology, route-context and tone evidence?
3. How do grouped counterfactual substitutions change raw score, calibrated `p_remove`, threshold crossings and page-global coherent `SAFE_REMOVE`?
4. Which spatial regions carry the largest negative SHAP contribution from each evidence family?

### Interpretation limits

- SHAP explains the **raw frozen 59R1 estimator**. The isotonic calibrator is analysed separately because its stepwise mapping can flatten different raw scores into the same `p_remove`.
- Grouped counterfactuals are diagnostic interventions, not causal estimates. Correlated predictors make group effects non-additive.
- “Attributable percentage” is reported only as a clearly labelled share of absolute negative SHAP mass and as counterfactual decision movement. It is not a scientific causal percentage.
- The current 60R2 output and frozen threshold remain unchanged.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pandas numpy pyarrow scikit-learn joblib shap matplotlib seaborn opencv-python-headless

## 1. Configuration and read-only output layout

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, sys, platform, random, math, warnings, zipfile

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
PAGE = 'Page258'

# Optional exact overrides. Use run roots, not bundle/ or page subdirectories.
SOURCE_60R2_RUN_OVERRIDE = None
REMOVE_59R1_RUN_OVERRIDE = None

SOURCE_60R2_ROOTS = [
    ROOT / 'notebook60r2_verified_integrated_runtime',
    ROOT / 'notebook60r2_integrated_runtime',
]
REMOVE_59R1_ROOTS = [
    ROOT / 'observable_residual_mask_m5_v8_7_deployment_freeze',
    ROOT / 'remove_deployment_freeze_v8_8',
    ROOT / 'remove_deployment_fit_and_freeze_v8_8',
]

OUTPUT_ROOT = ROOT / 'page258_shap_counterfactual_audit'
RUN = OUTPUT_ROOT / (datetime.now().strftime('%Y%m%d_%H%M%S') + '_page258_read_only_audit')
TABLES = RUN / 'tables'
VISUALS = RUN / 'visuals'
AUDIT = RUN / 'audit'
UPLOAD = RUN / 'upload'
for p in [RUN, TABLES, VISUALS, AUDIT, UPLOAD]:
    p.mkdir(parents=True, exist_ok=False)

SEED = 42
SHAP_BACKGROUND_ROWS = 512
SHAP_EXPLAIN_ROWS_PER_STRATUM = 1500
COUNTERFACTUAL_OBSERVED_BINS = [0, 64, 96, 128, 160, 192, 224, 256]
OVERLAY_CLIP_QUANTILE = 0.995

random.seed(SEED)
np.random.seed(SEED)


def require(condition, message):
    if not bool(condition):
        raise RuntimeError(message)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()


def read_json(path):
    path = Path(path)
    require(path.is_file(), f'Missing JSON: {path}')
    value = json.loads(path.read_text())
    require(isinstance(value, dict), f'JSON root is not an object: {path}')
    return value


def atomic_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, sort_keys=True, default=str))


def latest_complete(roots, override, required):
    if override is not None:
        run = Path(override)
        if run.name == 'bundle':
            run = run.parent
        require(all((run / rel).is_file() for rel in required), f'Incomplete override: {run}')
        return run
    hits = []
    for root in roots:
        if not root.exists():
            continue
        for candidate in root.iterdir():
            if candidate.is_dir() and all((candidate / rel).is_file() for rel in required):
                hits.append(candidate)
    require(hits, f'No complete run found under {[str(x) for x in roots]}')
    return max(hits, key=lambda p: p.stat().st_mtime)

print('Output:', RUN)

## 2. Resolve and freeze the source identities without modifying them

In [ ]:
SOURCE_60R2_REQUIRED = [
    'runtime_manifest.json',
    'verdict.json',
    'tables/pixel_predictions.parquet',
    f'pages/{PAGE}/page_manifest.json',
    f'pages/{PAGE}/runtime_arrays.npz',
    f'pages/{PAGE}/observed.png',
    f'pages/{PAGE}/safe_remove.png',
    f'pages/{PAGE}/m5_remove_mask.png',
]
REMOVE_REQUIRED = [
    'bundle/remove_model.joblib',
    'bundle/remove_calibrator.joblib',
    'bundle/predictor_schema.json',
    'bundle/policy.json',
    'bundle/freeze_manifest.json',
    'replay/replay_test.json',
]

SOURCE_60R2 = latest_complete(SOURCE_60R2_ROOTS, SOURCE_60R2_RUN_OVERRIDE, SOURCE_60R2_REQUIRED)
REMOVE_RUN = latest_complete(REMOVE_59R1_ROOTS, REMOVE_59R1_RUN_OVERRIDE, REMOVE_REQUIRED)
REMOVE_BUNDLE = REMOVE_RUN / 'bundle'

source_paths = {
    'runtime_manifest': SOURCE_60R2/'runtime_manifest.json',
    'verdict': SOURCE_60R2/'verdict.json',
    'pixel_predictions': SOURCE_60R2/'tables'/'pixel_predictions.parquet',
    'page_manifest': SOURCE_60R2/'pages'/PAGE/'page_manifest.json',
    'runtime_arrays': SOURCE_60R2/'pages'/PAGE/'runtime_arrays.npz',
    'observed': SOURCE_60R2/'pages'/PAGE/'observed.png',
    'safe_remove': SOURCE_60R2/'pages'/PAGE/'safe_remove.png',
    'm5_remove': SOURCE_60R2/'pages'/PAGE/'m5_remove_mask.png',
    'remove_model': REMOVE_BUNDLE/'remove_model.joblib',
    'remove_calibrator': REMOVE_BUNDLE/'remove_calibrator.joblib',
    'remove_schema': REMOVE_BUNDLE/'predictor_schema.json',
    'remove_policy': REMOVE_BUNDLE/'policy.json',
    'remove_freeze': REMOVE_BUNDLE/'freeze_manifest.json',
    'remove_replay': REMOVE_RUN/'replay'/'replay_test.json',
}
source_hashes = {key: sha256_file(path) for key, path in source_paths.items()}
atomic_json(AUDIT/'source_hashes_before.json', {k:{'path':str(source_paths[k]),'sha256':v} for k,v in source_hashes.items()})

runtime_manifest = read_json(source_paths['runtime_manifest'])
verdict = read_json(source_paths['verdict'])
page_manifest = read_json(source_paths['page_manifest'])
remove_schema = read_json(source_paths['remove_schema'])
remove_policy = read_json(source_paths['remove_policy'])
remove_freeze = read_json(source_paths['remove_freeze'])
remove_replay = read_json(source_paths['remove_replay'])

require(verdict.get('verdict') == 'PASS', 'Source 60R2 verdict is not PASS.')
require(remove_replay.get('passed') is True, 'Frozen 59R1 replay did not pass.')
require(runtime_manifest.get('production_authority') is False, 'Unexpected production-authority flag.')

predictors = remove_schema.get('ordered_predictors') or remove_schema.get('predictors')
require(isinstance(predictors, list) and len(predictors) == 17, 'Expected exact 17-field 59R1 schema.')

REMOVE_THRESHOLD = float(remove_policy.get('remove_threshold', remove_policy.get('threshold', 0.995)))
REMOVE_CONNECTIVITY = int(remove_policy.get('connectivity', 8))
REMOVE_MINIMUM_RUN = int(remove_policy.get('minimum_run', remove_policy.get('minimum_region_size', 2)))
REMOVE_REGION_MEAN_THRESHOLD = float(remove_policy.get('region_mean_threshold', REMOVE_THRESHOLD))

remove_model = joblib.load(source_paths['remove_model'])
remove_calibrator = joblib.load(source_paths['remove_calibrator'])
require(hasattr(remove_model, 'predict_proba'), 'Frozen 59R1 model lacks predict_proba().')
require(hasattr(remove_calibrator, 'predict'), 'Frozen 59R1 calibrator lacks predict().')

print('60R2 source:', SOURCE_60R2)
print('59R1 source:', REMOVE_RUN)
print('Threshold remains:', REMOVE_THRESHOLD)
print('Predictors:', predictors)

## 3. Load Page258 rows and reproduce the frozen scores and decisions exactly

In [ ]:
pixels = pd.read_parquet(source_paths['pixel_predictions'])
require({'page','y','x','raw_remove_probability','p_remove','safe_remove'}.issubset(pixels.columns),
        '60R2 pixel table lacks required output columns.')
page = pixels[pixels.page.astype(str) == PAGE].copy().reset_index(drop=True)
require(len(page) > 0, f'No {PAGE} rows in pixel_predictions.parquet.')
missing = [c for c in predictors if c not in page.columns]
require(not missing, f'Page258 table lacks frozen predictors: {missing}')

X = page[predictors].apply(pd.to_numeric, errors='coerce')
require(not np.isinf(X.to_numpy(float)).any(), 'Infinite predictor values found.')

raw_replay = np.asarray(remove_model.predict_proba(X)[:,1], np.float64)
p_replay = np.asarray(remove_calibrator.predict(raw_replay), np.float64)
require(np.array_equal(raw_replay, page.raw_remove_probability.to_numpy(np.float64)),
        'Raw 59R1 score replay is not exact.')
require(np.array_equal(p_replay, page.p_remove.to_numpy(np.float64)),
        'Calibrated p_remove replay is not exact.')

observed = cv2.imread(str(source_paths['observed']), cv2.IMREAD_GRAYSCALE)
safe_image = cv2.imread(str(source_paths['safe_remove']), cv2.IMREAD_GRAYSCALE) > 0
m5_image = cv2.imread(str(source_paths['m5_remove']), cv2.IMREAD_GRAYSCALE) > 0
require(observed is not None and safe_image.shape == observed.shape == m5_image.shape, 'Page image shape mismatch.')

inside = np.zeros(observed.shape, bool)
inside[page.y.to_numpy(int), page.x.to_numpy(int)] = True
safe_from_table = np.zeros(observed.shape, bool)
safe_from_table[page.y.to_numpy(int), page.x.to_numpy(int)] = page.safe_remove.astype(bool).to_numpy()
require(np.array_equal(safe_from_table, safe_image), 'Page258 SAFE_REMOVE table/image mismatch.')
require(np.array_equal(m5_image, inside & ~safe_image), 'Page258 M5_REMOVE identity mismatch.')

page['decision_group'] = np.where(page.safe_remove.astype(bool), 'SAFE_REMOVE', 'M5_RETAINED')
page['distance_to_threshold'] = page.p_remove.astype(float) - REMOVE_THRESHOLD
page.to_parquet(TABLES/'page258_frozen_rows.parquet', index=False)

print(page.decision_group.value_counts())
print('Raw/calibrated replay: EXACT')

## 4. Evidence-family definitions and descriptive contrasts

In [ ]:
GROUPS = {
    'p_print': ['p_print'],
    'boundary_mask_geometry': ['mask_depth','boundary_distance','component_id'],
    'protected_proximity': ['distance_to_protected','local_preserve_fraction'],
    'local_topology': [
        'edge','local_edge_fraction','local_dark_fraction',
        'local_horizontal_fraction','local_vertical_fraction','local_compact_fraction',
    ],
    'route_context': ['local_remove_fraction','local_ambiguous_fraction'],
    'tone': ['observed_value','normalised_value','paper_value'],
}

flat = [x for values in GROUPS.values() for x in values]
require(flat == predictors, 'Grouped feature order/coverage differs from frozen schema. Grouped=' + repr(flat) + ' Frozen=' + repr(predictors))

rows = []
for feature in predictors:
    a = page.loc[page.decision_group == 'M5_RETAINED', feature].astype(float)
    b = page.loc[page.decision_group == 'SAFE_REMOVE', feature].astype(float)
    pooled = math.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2) if len(a)>1 and len(b)>1 else np.nan
    smd = (a.mean() - b.mean()) / pooled if np.isfinite(pooled) and pooled > 0 else np.nan
    rows.append({
        'predictor': feature,
        'group': next(k for k,v in GROUPS.items() if feature in v),
        'm5_n': len(a), 'safe_n': len(b),
        'm5_mean': a.mean(), 'safe_mean': b.mean(),
        'm5_median': a.median(), 'safe_median': b.median(),
        'm5_q25': a.quantile(.25), 'm5_q75': a.quantile(.75),
        'safe_q25': b.quantile(.25), 'safe_q75': b.quantile(.75),
        'standardised_mean_difference_m5_minus_safe': smd,
    })
descriptive = pd.DataFrame(rows)
descriptive.to_csv(TABLES/'page258_predictor_comparison.csv', index=False)
display(descriptive.sort_values('standardised_mean_difference_m5_minus_safe', key=lambda s:s.abs(), ascending=False))

## 5. Model-aware SHAP on the frozen raw 59R1 estimator

In [ ]:
# Explain the raw estimator, not the isotonic calibrator.
# The pipeline is transformed using its fitted preprocessing steps when necessary.

def stratified_sample_indices(frame, n_per_stratum, seed=SEED):
    rng = np.random.default_rng(seed)
    selected = []
    # Retained pixels are divided by p_remove quartile; SAFE_REMOVE is a separate stratum.
    tmp = frame.copy()
    retained = tmp.decision_group == 'M5_RETAINED'
    tmp['score_band'] = 'SAFE_REMOVE'
    if retained.any():
        tmp.loc[retained, 'score_band'] = pd.qcut(
            tmp.loc[retained, 'p_remove'].rank(method='first'), 4,
            labels=['M5_Q1','M5_Q2','M5_Q3','M5_Q4']
        ).astype(str)
    for _, g in tmp.groupby('score_band'):
        idx = g.index.to_numpy()
        take = min(len(idx), n_per_stratum)
        selected.extend(rng.choice(idx, take, replace=False).tolist())
    return np.array(sorted(set(selected)), dtype=int)

explain_idx = stratified_sample_indices(page, SHAP_EXPLAIN_ROWS_PER_STRATUM)
background_idx = np.random.default_rng(SEED).choice(
    np.arange(len(page)), min(SHAP_BACKGROUND_ROWS, len(page)), replace=False
)

# Detect sklearn Pipeline while retaining exact fitted preprocessing.
if hasattr(remove_model, 'steps'):
    preprocessing = remove_model[:-1]
    estimator = remove_model.steps[-1][1]
    X_transformed = preprocessing.transform(X)
    X_explain_t = X_transformed[explain_idx]
    X_background_t = X_transformed[background_idx]
else:
    preprocessing = None
    estimator = remove_model
    X_transformed = X.to_numpy(float)
    X_explain_t = X_transformed[explain_idx]
    X_background_t = X_transformed[background_idx]

estimator_name = estimator.__class__.__name__
print('Estimator:', estimator_name)

# Prefer model-specific exact/fast explainers. Fall back to permutation SHAP on the complete pipeline.
TREE_NAMES = {'RandomForestClassifier','ExtraTreesClassifier','GradientBoostingClassifier',
              'HistGradientBoostingClassifier','XGBClassifier','LGBMClassifier','CatBoostClassifier'}
LINEAR_NAMES = {'LogisticRegression','RidgeClassifier','SGDClassifier'}

if estimator_name in TREE_NAMES:
    explainer = shap.TreeExplainer(estimator, data=X_background_t, feature_names=predictors,
                                   feature_perturbation='interventional', model_output='probability')
    sv = explainer(X_explain_t, check_additivity=False)
    values = sv.values
    base_values = sv.base_values
    if values.ndim == 3:
        values = values[:,:,1]
        base_values = base_values[:,1] if np.ndim(base_values)==2 else base_values
    explanation_method = 'TreeExplainer_probability_interventional'
elif estimator_name in LINEAR_NAMES:
    explainer = shap.LinearExplainer(estimator, X_background_t, feature_names=predictors)
    sv = explainer(X_explain_t)
    values = sv.values
    base_values = sv.base_values
    explanation_method = 'LinearExplainer_raw_model_space'
else:
    background = X.iloc[background_idx]
    foreground = X.iloc[explain_idx]
    predict_raw = lambda data: np.asarray(remove_model.predict_proba(pd.DataFrame(data, columns=predictors))[:,1])
    masker = shap.maskers.Independent(background, max_samples=len(background))
    explainer = shap.Explainer(predict_raw, masker, algorithm='permutation', feature_names=predictors)
    sv = explainer(foreground, max_evals=2*len(predictors)+2048)
    values = sv.values
    base_values = sv.base_values
    explanation_method = 'PermutationExplainer_pipeline_probability'

require(values.shape == (len(explain_idx), len(predictors)), f'Unexpected SHAP shape: {values.shape}')

shap_rows = page.loc[explain_idx, ['page','y','x','decision_group','p_remove','raw_remove_probability']].reset_index(drop=True)
for j, feature in enumerate(predictors):
    shap_rows[f'shap_{feature}'] = values[:,j]
    shap_rows[f'value_{feature}'] = X.iloc[explain_idx, j].to_numpy()
shap_rows['shap_base_value'] = np.asarray(base_values).reshape(-1) if np.asarray(base_values).size==len(explain_idx) else float(np.asarray(base_values).reshape(-1)[0])
shap_rows.to_parquet(TABLES/'page258_pixel_shap.parquet', index=False)

metadata = {
    'estimator_class': estimator_name,
    'explanation_method': explanation_method,
    'rows_explained': len(explain_idx),
    'background_rows': len(background_idx),
    'predictors': predictors,
    'threshold_unchanged': REMOVE_THRESHOLD,
}
atomic_json(AUDIT/'shap_method.json', metadata)
print(json.dumps(metadata, indent=2))

## 6. Aggregate negative SHAP pressure by predictor and evidence family

In [ ]:
# Negative SHAP values reduce the raw REMOVE score in the explainer's output space.
long_rows = []
for j, feature in enumerate(predictors):
    for decision in ['M5_RETAINED','SAFE_REMOVE']:
        hit = shap_rows.decision_group.eq(decision).to_numpy()
        v = values[hit,j]
        long_rows.append({
            'predictor': feature,
            'group': next(k for k,g in GROUPS.items() if feature in g),
            'decision_group': decision,
            'rows': int(hit.sum()),
            'mean_shap': float(np.mean(v)),
            'median_shap': float(np.median(v)),
            'mean_negative_shap': float(np.mean(np.minimum(v,0))),
            'absolute_negative_shap_mass': float(np.abs(np.minimum(v,0)).sum()),
            'mean_absolute_shap': float(np.abs(v).mean()),
        })
feature_attr = pd.DataFrame(long_rows)
feature_attr.to_csv(TABLES/'page258_feature_shap_summary.csv', index=False)

retained = shap_rows.decision_group.eq('M5_RETAINED').to_numpy()
group_rows = []
absolute_negative_total = float(np.abs(np.minimum(values[retained],0)).sum())
for group, features in GROUPS.items():
    idx = [predictors.index(f) for f in features]
    grouped = values[retained][:,idx].sum(axis=1)
    negative_mass = float(np.abs(np.minimum(values[retained][:,idx],0)).sum())
    group_rows.append({
        'group': group,
        'features': '|'.join(features),
        'retained_rows_explained': int(retained.sum()),
        'mean_group_shap': float(grouped.mean()),
        'median_group_shap': float(np.median(grouped)),
        'mean_negative_group_shap': float(np.minimum(grouped,0).mean()),
        'absolute_negative_shap_mass': negative_mass,
        'share_of_absolute_negative_shap_mass': negative_mass / absolute_negative_total if absolute_negative_total else np.nan,
    })
group_attr = pd.DataFrame(group_rows).sort_values('absolute_negative_shap_mass', ascending=False)
group_attr.to_csv(TABLES/'page258_grouped_attribution.csv', index=False)
display(group_attr)

plt.figure(figsize=(10,6))
q = group_attr.sort_values('share_of_absolute_negative_shap_mass')
plt.barh(q.group, q.share_of_absolute_negative_shap_mass)
plt.xlabel('Share of absolute negative SHAP mass on sampled M5-retained pixels')
plt.title('Page258: evidence families reducing raw REMOVE score')
plt.tight_layout()
plt.savefig(VISUALS/'page258_negative_shap_share_by_group.png', dpi=180)
plt.close()

# Standard SHAP summary plot.
shap.summary_plot(values, X.iloc[explain_idx], feature_names=predictors, show=False, max_display=17)
plt.tight_layout()
plt.savefig(VISUALS/'page258_shap_summary.png', dpi=180, bbox_inches='tight')
plt.close()

## 7. Read-only calibration mapping audit

In [ ]:
order = np.argsort(raw_replay)
calibration = pd.DataFrame({'raw_remove_probability': raw_replay[order], 'p_remove': p_replay[order]})
calibration = calibration.drop_duplicates().reset_index(drop=True)
calibration.to_csv(TABLES/'page258_calibration_mapping.csv', index=False)

plt.figure(figsize=(8,6))
plt.step(calibration.raw_remove_probability, calibration.p_remove, where='post')
plt.axhline(REMOVE_THRESHOLD, color='red', linestyle='--', label=f'frozen threshold {REMOVE_THRESHOLD}')
plt.xlabel('Raw frozen 59R1 REMOVE score')
plt.ylabel('Calibrated p_remove')
plt.title('Page258: isotonic calibration mapping over encountered scores')
plt.legend()
plt.tight_layout()
plt.savefig(VISUALS/'page258_calibration_mapping.png', dpi=180)
plt.close()

## 8. Frozen page-global coherence helper

In [ ]:
def coherent_safe_remove(frame, p_remove, shape):
    provisional = np.asarray(p_remove) >= REMOVE_THRESHOLD
    candidate = np.zeros(shape, np.uint8)
    candidate[frame.loc[provisional,'y'].to_numpy(int), frame.loc[provisional,'x'].to_numpy(int)] = 1
    n, labels, stats, _ = cv2.connectedComponentsWithStats(candidate, REMOVE_CONNECTIVITY)
    probability_map = np.full(shape, np.nan, np.float32)
    probability_map[frame.y.to_numpy(int), frame.x.to_numpy(int)] = np.asarray(p_remove, np.float32)
    safe = np.zeros(shape, bool)
    region_rows = []
    for cid in range(1,n):
        region = labels == cid
        probs = probability_map[region]
        finite = probs[np.isfinite(probs)]
        mean = float(finite.mean()) if len(finite) else np.nan
        minimum = float(finite.min()) if len(finite) else np.nan
        passed = bool(
            int(region.sum()) >= REMOVE_MINIMUM_RUN
            and len(finite) == int(region.sum())
            and mean >= REMOVE_REGION_MEAN_THRESHOLD
            and minimum >= REMOVE_THRESHOLD
        )
        if passed:
            safe[region] = True
        region_rows.append({'region_id':cid,'pixels':int(region.sum()),'mean_p_remove':mean,
                            'minimum_p_remove':minimum,'safe_remove':passed})
    return safe, probability_map, pd.DataFrame(region_rows)

safe_replay, _, _ = coherent_safe_remove(page, p_replay, observed.shape)
require(np.array_equal(safe_replay, safe_image), 'Coherence helper does not reproduce Page258 SAFE_REMOVE exactly.')
print('Frozen coherence replay: EXACT')

## 9. Grouped counterfactual audit

In [ ]:
# Reference A: Page258 SAFE_REMOVE medians stratified by observed-value band.
# Reference B: overall Page258 SAFE_REMOVE medians as fallback.
page_cf = page.copy()
page_cf['observed_band'] = pd.cut(page_cf.observed_value, COUNTERFACTUAL_OBSERVED_BINS,
                                  right=False, include_lowest=True, duplicates='drop')
safe_reference = page_cf[page_cf.safe_remove.astype(bool)]
require(len(safe_reference) > 0, 'Page258 contains no SAFE_REMOVE reference pixels.')
band_reference = safe_reference.groupby('observed_band', observed=True)[predictors].median(numeric_only=True)
overall_reference = safe_reference[predictors].median(numeric_only=True)


def apply_group_reference(X_original, group_features):
    changed = X_original.copy()
    bands = page_cf.observed_band.reset_index(drop=True)
    for feature in group_features:
        mapped = bands.map(band_reference[feature] if feature in band_reference else {})
        changed[feature] = mapped.astype(float).fillna(float(overall_reference[feature]))
    return changed


def score_counterfactual(name, features):
    X_cf = apply_group_reference(X.copy(), features)
    raw_cf = np.asarray(remove_model.predict_proba(X_cf)[:,1], np.float64)
    p_cf = np.asarray(remove_calibrator.predict(raw_cf), np.float64)
    safe_cf, _, regions_cf = coherent_safe_remove(page, p_cf, observed.shape)
    retained = ~safe_image[page.y.to_numpy(int), page.x.to_numpy(int)]
    return {
        'name': name,
        'features': '|'.join(features),
        'raw_cf': raw_cf,
        'p_cf': p_cf,
        'safe_cf': safe_cf,
        'regions_cf': regions_cf,
        'summary': {
            'counterfactual': name,
            'features': '|'.join(features),
            'mean_delta_raw_all': float(np.mean(raw_cf - raw_replay)),
            'mean_delta_p_remove_all': float(np.mean(p_cf - p_replay)),
            'mean_delta_raw_m5_retained': float(np.mean((raw_cf - raw_replay)[retained])),
            'mean_delta_p_remove_m5_retained': float(np.mean((p_cf - p_replay)[retained])),
            'pixels_crossing_up_over_0995': int(np.sum((p_replay < REMOVE_THRESHOLD) & (p_cf >= REMOVE_THRESHOLD))),
            'pixels_crossing_down_below_0995': int(np.sum((p_replay >= REMOVE_THRESHOLD) & (p_cf < REMOVE_THRESHOLD))),
            'safe_remove_pixels_baseline': int(safe_image.sum()),
            'safe_remove_pixels_counterfactual': int(safe_cf.sum()),
            'safe_remove_pixel_delta': int(safe_cf.sum() - safe_image.sum()),
            'm5_pixels_baseline': int(m5_image.sum()),
            'm5_pixels_counterfactual': int((inside & ~safe_cf).sum()),
            'm5_pixel_delta': int((inside & ~safe_cf).sum() - m5_image.sum()),
        }
    }

counterfactuals = []
for group, features in GROUPS.items():
    counterfactuals.append(score_counterfactual(group, features))

# Combined non-p_print structural counterfactual and all-feature matched reference.
non_pprint_features = [f for f in predictors if f != 'p_print']
counterfactuals.append(score_counterfactual('all_non_p_print_evidence', non_pprint_features))
counterfactuals.append(score_counterfactual('all_predictors', predictors))

cf_summary = pd.DataFrame([x['summary'] for x in counterfactuals])
cf_summary.to_csv(TABLES/'page258_counterfactual_ablation.csv', index=False)
display(cf_summary.sort_values('safe_remove_pixel_delta', ascending=False))

threshold_rows = []
for result in counterfactuals:
    delta = result['p_cf'] - p_replay
    for group_name, hit in [('M5_RETAINED', ~page.safe_remove.astype(bool).to_numpy()),
                            ('SAFE_REMOVE', page.safe_remove.astype(bool).to_numpy())]:
        threshold_rows.append({
            'counterfactual': result['name'], 'population': group_name,
            'rows': int(hit.sum()), 'mean_delta_p_remove': float(delta[hit].mean()),
            'median_delta_p_remove': float(np.median(delta[hit])),
            'q95_delta_p_remove': float(np.quantile(delta[hit],.95)),
            'crossing_up': int(np.sum(hit & (p_replay < REMOVE_THRESHOLD) & (result['p_cf'] >= REMOVE_THRESHOLD))),
            'crossing_down': int(np.sum(hit & (p_replay >= REMOVE_THRESHOLD) & (result['p_cf'] < REMOVE_THRESHOLD))),
        })
pd.DataFrame(threshold_rows).to_csv(TABLES/'page258_threshold_crossings.csv', index=False)

## 10. Spatial overlays and `p_print` versus `p_remove` diagnostics

In [ ]:
def contribution_map(group):
    features = GROUPS[group]
    indices = [predictors.index(f) for f in features]
    values_group = values[:,indices].sum(axis=1)
    image = np.full(observed.shape, np.nan, np.float32)
    image[shap_rows.y.to_numpy(int), shap_rows.x.to_numpy(int)] = values_group.astype(np.float32)
    return image


def save_signed_overlay(values_map, path, title):
    finite = np.isfinite(values_map)
    clip = float(np.quantile(np.abs(values_map[finite]), OVERLAY_CLIP_QUANTILE)) if finite.any() else 1.0
    clip = max(clip, 1e-9)
    norm = np.clip((values_map + clip) / (2*clip), 0, 1)
    colour = cv2.applyColorMap((np.nan_to_num(norm, nan=.5)*255).astype(np.uint8), cv2.COLORMAP_COOLWARM)
    base = cv2.cvtColor(observed, cv2.COLOR_GRAY2BGR)
    out = base.copy()
    active = finite
    out[active] = cv2.addWeighted(base, .35, colour, .65, 0)[active]
    cv2.imwrite(str(path), out)

for group in GROUPS:
    save_signed_overlay(contribution_map(group), VISUALS/f'page258_shap_overlay_{group}.png', group)

plt.figure(figsize=(9,7))
plot_df = page.sample(min(50000,len(page)), random_state=SEED)
sns.scatterplot(data=plot_df, x='p_print', y='p_remove', hue='decision_group', alpha=.25, s=8)
plt.axhline(REMOVE_THRESHOLD, color='red', linestyle='--')
plt.title('Page258: continuous p_print versus calibrated p_remove')
plt.tight_layout()
plt.savefig(VISUALS/'page258_pprint_vs_premove.png', dpi=180)
plt.close()

plt.figure(figsize=(11,6))
q = cf_summary.sort_values('safe_remove_pixel_delta')
plt.barh(q.counterfactual, q.safe_remove_pixel_delta)
plt.axvline(0, color='black', linewidth=1)
plt.xlabel('Change in coherent SAFE_REMOVE pixels')
plt.title('Page258 grouped counterfactual decision movement')
plt.tight_layout()
plt.savefig(VISUALS/'page258_counterfactual_safe_remove_delta.png', dpi=180)
plt.close()

## 11. Read-only verification and report

In [ ]:
# Verify every source file is unchanged after the audit.
source_hashes_after = {key: sha256_file(path) for key, path in source_paths.items()}
require(source_hashes_after == source_hashes, 'A source artefact changed during the read-only audit.')
atomic_json(AUDIT/'source_hashes_after.json', {k:{'path':str(source_paths[k]),'sha256':v} for k,v in source_hashes_after.items()})

pprint_row = group_attr[group_attr.group == 'p_print'].iloc[0]
structural_row = cf_summary[cf_summary.counterfactual == 'all_non_p_print_evidence'].iloc[0]
pprint_cf = cf_summary[cf_summary.counterfactual == 'p_print'].iloc[0]

report = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'page': PAGE,
    'read_only': True,
    'source_60r2': str(SOURCE_60R2),
    'source_59r1': str(REMOVE_RUN),
    'threshold_unchanged': REMOVE_THRESHOLD,
    'model_sha256': source_hashes['remove_model'],
    'calibrator_sha256': source_hashes['remove_calibrator'],
    'schema_sha256': source_hashes['remove_schema'],
    'source_hashes_unchanged': True,
    'raw_score_replay_exact': True,
    'calibrated_probability_replay_exact': True,
    'coherent_safe_remove_replay_exact': True,
    'shap_method': explanation_method,
    'shap_rows': len(explain_idx),
    'p_print_share_of_absolute_negative_shap_mass_on_sampled_m5': float(pprint_row.share_of_absolute_negative_shap_mass),
    'p_print_counterfactual_safe_remove_delta': int(pprint_cf.safe_remove_pixel_delta),
    'all_non_p_print_counterfactual_safe_remove_delta': int(structural_row.safe_remove_pixel_delta),
    'causal_claim_permitted': False,
    'production_authority': False,
}
atomic_json(RUN/'page258_attribution_report.json', report)

lines = [
    'PAGE258 READ-ONLY SHAP AND GROUPED COUNTERFACTUAL AUDIT',
    '',
    f'60R2 source: {SOURCE_60R2}',
    f'59R1 source: {REMOVE_RUN}',
    f'Frozen threshold: {REMOVE_THRESHOLD}',
    f'Estimator: {estimator_name}',
    f'SHAP method: {explanation_method}',
    f'Rows explained: {len(explain_idx)}',
    '',
    f'p_print share of absolute negative SHAP mass on sampled retained pixels: {float(pprint_row.share_of_absolute_negative_shap_mass):.6f}',
    f'Coherent SAFE_REMOVE pixel change when p_print is replaced by matched SAFE_REMOVE references: {int(pprint_cf.safe_remove_pixel_delta)}',
    f'Coherent SAFE_REMOVE pixel change when all non-p_print evidence is replaced: {int(structural_row.safe_remove_pixel_delta)}',
    '',
    'These quantities are model diagnostics, not causal percentages. Correlated predictors and isotonic calibration make grouped effects non-additive.',
    'No source output, model, calibrator, schema, threshold or Page258 decision array was modified.',
]
(RUN/'executive_summary.txt').write_text(chr(10).join(lines))
print(chr(10).join(lines))


## 12. Curated final upload package

In [ ]:
curated = [
    RUN/'executive_summary.txt',
    RUN/'page258_attribution_report.json',
    AUDIT/'source_hashes_before.json',
    AUDIT/'source_hashes_after.json',
    AUDIT/'shap_method.json',
    TABLES/'page258_predictor_comparison.csv',
    TABLES/'page258_feature_shap_summary.csv',
    TABLES/'page258_grouped_attribution.csv',
    TABLES/'page258_counterfactual_ablation.csv',
    TABLES/'page258_threshold_crossings.csv',
    TABLES/'page258_calibration_mapping.csv',
    VISUALS/'page258_shap_summary.png',
    VISUALS/'page258_negative_shap_share_by_group.png',
    VISUALS/'page258_pprint_vs_premove.png',
    VISUALS/'page258_counterfactual_safe_remove_delta.png',
]
curated += sorted(VISUALS.glob('page258_shap_overlay_*.png'))

manifest = []
for src in curated:
    require(src.is_file(), f'Missing curated output: {src}')
    rel = src.relative_to(RUN)
    dst = UPLOAD / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    manifest.append({'relative_path':str(rel),'bytes':src.stat().st_size,'sha256':sha256_file(src)})

pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv', index=False)
readme_lines = ['Page258 read-only SHAP and grouped counterfactual audit.', 'Source 60R2 and 59R1 artefacts remain in place and were hash-verified before and after analysis.', 'The frozen threshold and original SAFE_REMOVE/M5 decisions were not changed.']
(UPLOAD/'README.txt').write_text(chr(10).join(readme_lines) + chr(10))
(UPLOAD/'upload_file_list.txt').write_text(chr(10).join(sorted(str(p.relative_to(UPLOAD)) for p in UPLOAD.rglob('*') if p.is_file())) + chr(10))

zip_path = RUN/'60R3_Page258_Read_Only_SHAP_and_Grouped_Counterfactual_Audit_upload_bundle.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file():
            zf.write(p, p.relative_to(UPLOAD))
require(zip_path.is_file() and zip_path.stat().st_size > 0, 'Upload ZIP creation failed.')
print('Upload ZIP:', zip_path)